# Newton root-finder for ppm_Boron using OpenMC tally derivatives (one run per Newton iter).

In [ ]:
#!/usr/bin/env python3
"""
gradient_optimization_demo.py - Demonstrating gradient-based optimization speedup with plotting
"""

import os
import math
import h5py
import openmc
import numpy as np
import warnings
import matplotlib.pyplot as plt

# Suppress FutureWarnings for cleaner output
warnings.filterwarnings('ignore', category=FutureWarning)

# Constants
N_A = 6.02214076e23     # Avogadro's number (atoms/mol)
A_B_nat = 10.81         # g/mol approximate atomic mass for natural boron

# ===============================================================
# Model builder
# ===============================================================
def build_model(ppm_Boron):
    # Create the pin materials
    fuel = openmc.Material(name='1.6% Fuel', material_id=1)
    fuel.set_density('g/cm3', 10.31341)
    fuel.add_element('U', 1., enrichment=1.6)
    fuel.add_element('O', 2.)

    zircaloy = openmc.Material(name='Zircaloy', material_id=2)
    zircaloy.set_density('g/cm3', 6.55)
    zircaloy.add_element('Zr', 1.)

    water = openmc.Material(name='Borated Water', material_id=3)
    water.set_density('g/cm3', 0.741)
    water.add_element('H', 2.)
    water.add_element('O', 1.)
    water.add_element('B', ppm_Boron * 1e-6)

    materials = openmc.Materials([fuel, zircaloy, water])

    # Geometry
    fuel_outer_radius = openmc.ZCylinder(r=0.39218)
    clad_outer_radius = openmc.ZCylinder(r=0.45720)

    min_x = openmc.XPlane(x0=-0.63, boundary_type='reflective')
    max_x = openmc.XPlane(x0=+0.63, boundary_type='reflective')
    min_y = openmc.YPlane(y0=-0.63, boundary_type='reflective')
    max_y = openmc.YPlane(y0=+0.63, boundary_type='reflective')

    fuel_cell = openmc.Cell(name='1.6% Fuel')
    fuel_cell.fill = fuel
    fuel_cell.region = -fuel_outer_radius

    clad_cell = openmc.Cell(name='1.6% Clad')
    clad_cell.fill = zircaloy
    clad_cell.region = +fuel_outer_radius & -clad_outer_radius

    moderator_cell = openmc.Cell(name='1.6% Moderator')
    moderator_cell.fill = water
    moderator_cell.region = +clad_outer_radius & (+min_x & -max_x & +min_y & -max_y)

    root_universe = openmc.Universe(name='root universe', universe_id=0)
    root_universe.add_cells([fuel_cell, clad_cell, moderator_cell])

    geometry = openmc.Geometry(root_universe)

    # Settings
    settings = openmc.Settings()
    settings.batches = 50
    settings.inactive = 10
    settings.particles = 1000
    settings.run_mode = 'eigenvalue'

    bounds = [-0.63, -0.63, -10, 0.63, 0.63, 10.]
    uniform_dist = openmc.stats.Box(bounds[:3], bounds[3:], only_fissionable=True)
    settings.source = openmc.Source(space=uniform_dist)

    model = openmc.model.Model(geometry, materials, settings)
    return model

# ===============================================================
# Helper: automatically find all cell IDs filled with a material
# ===============================================================
def find_cells_using_material(geometry, material):
    return [c.id for c in geometry.get_all_cells().values() if c.fill is material]

# ===============================================================
# Run OpenMC with gradient calculation
# ===============================================================
def run_with_gradient(ppm_B, target_batches=50, water_material_id=3, boron_nuclides=('B10', 'B11')):
    """Run OpenMC and compute k-effective with gradient information"""
    # Clean up previous files
    for f in ['summary.h5', f'statepoint.{target_batches}.h5', 'tallies.out']:
        if os.path.exists(f):
            os.remove(f)
    
    # Build model
    model = build_model(ppm_B)
    
    # Auto-detect moderator cells
    water = model.materials[water_material_id - 1]  # Materials are 0-indexed
    moderator_cell_ids = find_cells_using_material(model.geometry, water)
    moderator_filter = openmc.CellFilter(moderator_cell_ids)

    # Base tallies
    tF_base = openmc.Tally(name='FissionBase')
    tF_base.scores = ['nu-fission']
    tA_base = openmc.Tally(name='AbsorptionBase')
    tA_base.scores = ['absorption']

    # Derivative tallies
    deriv_tallies = []
    for nuc in boron_nuclides:
        deriv = openmc.TallyDerivative(
            variable='nuclide_density',
            material=water_material_id,
            nuclide=nuc
        )

        tf = openmc.Tally(name=f'Fission_deriv_{nuc}')
        tf.scores = ['nu-fission']
        tf.derivative = deriv
        tf.filters = [moderator_filter]

        ta = openmc.Tally(name=f'Absorp_deriv_{nuc}')
        ta.scores = ['absorption']
        ta.derivative = deriv
        ta.filters = [moderator_filter]

        deriv_tallies += [tf, ta]

    model.tallies = openmc.Tallies([tF_base, tA_base] + deriv_tallies)
    model.settings.batches = target_batches
    model.settings.inactive = max(1, int(target_batches * 0.1))

    # Run simulation
    model.run()
    sp = openmc.StatePoint(f"statepoint.{target_batches}.h5")
    
    # Get results
    k_eff = sp.keff.nominal_value
    
    # Base tallies
    fission_tally = sp.get_tally(name='FissionBase')
    absorption_tally = sp.get_tally(name='AbsorptionBase')
    F_base = float(np.sum(fission_tally.mean))
    A_base = float(np.sum(absorption_tally.mean))
    
    # Derivative tallies
    dF_dN_total = 0.0
    dA_dN_total = 0.0
    
    for nuc in boron_nuclides:
        fission_deriv = sp.get_tally(name=f'Fission_deriv_{nuc}')
        absorption_deriv = sp.get_tally(name=f'Absorp_deriv_{nuc}')
        
        if fission_deriv:
            dF_dN_total += float(np.sum(fission_deriv.mean))
        if absorption_deriv:
            dA_dN_total += float(np.sum(absorption_deriv.mean))
    
    return k_eff, F_base, A_base, dF_dN_total, dA_dN_total, water.density

# ===============================================================
# Run OpenMC without gradient (for comparison)
# ===============================================================
def run_without_gradient(ppm_B, target_batches=50):
    """Run OpenMC without gradient calculation (for comparison)"""
    # Clean up previous files
    for f in ['summary.h5', f'statepoint.{target_batches}.h5', 'tallies.out']:
        if os.path.exists(f):
            os.remove(f)
    
    model = build_model(ppm_B)
    model.settings.batches = target_batches
    model.settings.inactive = max(1, int(target_batches * 0.1))
    
    model.run()
    sp = openmc.StatePoint(f"statepoint.{target_batches}.h5")
    return sp.keff.nominal_value

# ===============================================================
# Gradient-Based Optimization
# ===============================================================
def gradient_based_search(ppm_start, k_target, tol=1e-3, max_iter=8, learning_rate=1e-14):
    """Gradient-based optimization using analytical derivatives"""
    ppm = float(ppm_start)
    history = []
    
    print("GRADIENT-BASED OPTIMIZATION")
    print(f"Initial: {ppm:.1f} ppm, Target: k = {k_target}")
    print("Iter |   ppm   |   k_eff   |  Error  |  Gradient  |  Step")
    print("-" * 65)
    
    for it in range(max_iter):
        k, F, A, dF_dN, dA_dN, rho_water = run_with_gradient(ppm)
        err = k - k_target
        history.append((ppm, k, err, dF_dN, dA_dN))
        
        # Calculate gradient using chain rule
        dk_dN = (A * dF_dN - F * dA_dN) / (A * A)
        dN_dppm = 1e-6 * rho_water * N_A / A_B_nat
        dk_dppm = dk_dN * dN_dppm
        
        # Gradient descent step
        step = -learning_rate * err * dk_dppm
        ppm_new = ppm + step
        
        # Apply bounds
        ppm_new = max(500.0, min(ppm_new, 3000.0))
        
        print(f"{it+1:3d} | {ppm:7.1f} | {k:9.6f} | {err:7.4f} | {dk_dppm:10.2e} | {step:7.1f}")
        
        if abs(err) < tol:
            print(f"✓ CONVERGED in {it+1} iterations")
            return ppm, history
        
        ppm = ppm_new
    
    print(f"Reached maximum iterations ({max_iter})")
    return ppm, history

# ===============================================================
# Gradient-Free Optimization (for comparison)
# ===============================================================
def gradient_free_search(ppm_start, k_target, tol=1e-3, max_iter=12):
    """Gradient-free optimization using heuristic steps"""
    ppm = float(ppm_start)
    history = []
    
    print("\nGRADIENT-FREE OPTIMIZATION")
    print(f"Initial: {ppm:.1f} ppm, Target: k = {k_target}")
    print("Iter |   ppm   |   k_eff   |  Error  |   Step")
    print("-" * 55)
    
    for it in range(max_iter):
        k = run_without_gradient(ppm)
        err = k - k_target
        history.append((ppm, k, err, 0, 0))  # zeros for derivatives for consistency
        
        # Adaptive step size based on error
        if abs(err) > 0.1:
            step = 300
        elif abs(err) > 0.05:
            step = 200
        else:
            step = 100
        
        # Determine direction
        if err > 0:  # k too high, need more boron
            ppm_new = ppm + step
            step_str = f"+{step}"
        else:  # k too low, need less boron
            ppm_new = ppm - step
            step_str = f"-{step}"
        
        # Apply bounds
        ppm_new = max(500.0, min(ppm_new, 3000.0))
        
        print(f"{it+1:3d} | {ppm:7.1f} | {k:9.6f} | {err:7.4f} | {step_str:>7}")
        
        if abs(err) < tol:
            print(f"✓ CONVERGED in {it+1} iterations")
            return ppm, history
        
        ppm = ppm_new
    
    print(f"Reached maximum iterations ({max_iter})")
    return ppm, history

# ===============================================================
# Finite Difference Gradient (alternative approach)
# ===============================================================
def finite_difference_search(ppm_start, k_target, tol=1e-3, max_iter=8, perturbation=100):
    """Optimization using finite difference gradients"""
    ppm = float(ppm_start)
    history = []
    
    print("\nFINITE DIFFERENCE OPTIMIZATION")
    print(f"Initial: {ppm:.1f} ppm, Target: k = {k_target}")
    print("Iter |   ppm   |   k_eff   |  Error  |  FD Gradient |  Step")
    print("-" * 70)
    
    for it in range(max_iter):
        k_current = run_without_gradient(ppm)
        err = k_current - k_target
        history.append((ppm, k_current, err, 0, 0))  # zeros for derivatives for consistency
        
        # Finite difference gradient
        ppm_perturbed = ppm + perturbation
        k_perturbed = run_without_gradient(ppm_perturbed)
        dk_dppm_fd = (k_perturbed - k_current) / perturbation
        
        # Gradient descent step (careful with step size)
        if abs(dk_dppm_fd) > 1e-10:
            step = -0.1 * err / dk_dppm_fd  # Conservative step
        else:
            # Fallback heuristic
            step = 200 if err > 0 else -200
        
        ppm_new = ppm + step
        ppm_new = max(500.0, min(ppm_new, 3000.0))
        
        print(f"{it+1:3d} | {ppm:7.1f} | {k_current:9.6f} | {err:7.4f} | {dk_dppm_fd:12.2e} | {step:7.1f}")
        
        if abs(err) < tol:
            print(f"✓ CONVERGED in {it+1} iterations")
            return ppm, history
        
        ppm = ppm_new
    
    print(f"Reached maximum iterations ({max_iter})")
    return ppm, history

# ===============================================================
# Plotting Functions
# ===============================================================
def plot_convergence_comparison(methods_data, k_target, output_file='convergence_comparison.png'):
    """Plot convergence comparison for all methods"""
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 6))
    
    colors = {'Analytical Gradient': 'blue', 'Finite Difference': 'green', 'Gradient-Free': 'red'}
    markers = {'Analytical Gradient': 'o', 'Finite Difference': 's', 'Gradient-Free': '^'}
    
    # Plot 1: Error vs Iterations
    for name, ppm, history in methods_data:
        if history:
            iterations = range(1, len(history) + 1)
            errors = [abs(h[2]) for h in history]  # Absolute error
            ax1.semilogy(iterations, errors, marker=markers[name], color=colors[name], 
                        label=name, linewidth=2, markersize=8)
    
    ax1.set_xlabel('Iteration')
    ax1.set_ylabel('Absolute Error |k - k_target|')
    ax1.set_title('Convergence: Error vs Iterations')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Plot 2: Boron Concentration vs Iterations
    for name, ppm, history in methods_data:
        if history:
            iterations = range(1, len(history) + 1)
            concentrations = [h[0] for h in history]  # Boron concentrations
            ax2.plot(iterations, concentrations, marker=markers[name], color=colors[name],
                    label=name, linewidth=2, markersize=8)
    
    ax2.set_xlabel('Iteration')
    ax2.set_ylabel('Boron Concentration (ppm)')
    ax2.set_title('Boron Concentration Evolution')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # Plot 3: k-effective vs Iterations
    for name, ppm, history in methods_data:
        if history:
            iterations = range(1, len(history) + 1)
            k_effs = [h[1] for h in history]  # k-effective values
            ax3.plot(iterations, k_effs, marker=markers[name], color=colors[name],
                    label=name, linewidth=2, markersize=8)
    
    # Add target line
    ax3.axhline(y=k_target, color='black', linestyle='--', alpha=0.7, label=f'Target (k={k_target})')
    ax3.set_xlabel('Iteration')
    ax3.set_ylabel('k-effective')
    ax3.set_title('k-effective Evolution')
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    print(f"\nConvergence plots saved as '{output_file}'")
    plt.show()

def plot_detailed_convergence(methods_data, k_target, output_file='detailed_convergence.png'):
    """Create detailed convergence plots with step information"""
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 10))
    
    colors = {'Analytical Gradient': 'blue', 'Finite Difference': 'green', 'Gradient-Free': 'red'}
    
    # Plot 1: Error convergence
    for name, ppm, history in methods_data:
        if history:
            iterations = range(1, len(history) + 1)
            errors = [abs(h[2]) for h in history]
            ax1.semilogy(iterations, errors, 'o-', color=colors[name], label=name, linewidth=2)
    
    ax1.set_xlabel('Iteration')
    ax1.set_ylabel('Absolute Error')
    ax1.set_title('Error Convergence (Log Scale)')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Plot 2: Boron concentration steps
    for name, ppm, history in methods_data:
        if history:
            iterations = range(len(history))
            concentrations = [h[0] for h in history]
            # Plot lines between points to show steps
            for i in range(len(concentrations)-1):
                ax2.plot([iterations[i], iterations[i+1]], [concentrations[i], concentrations[i+1]], 
                        'o-', color=colors[name], linewidth=2, markersize=6,
                        label=name if i == 0 else "")
    
    ax2.set_xlabel('Iteration')
    ax2.set_ylabel('Boron Concentration (ppm)')
    ax2.set_title('Boron Concentration Steps')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # Plot 3: Relative error (percentage)
    for name, ppm, history in methods_data:
        if history:
            iterations = range(1, len(history) + 1)
            relative_errors = [abs(h[2]/k_target * 100) for h in history]  # Percentage
            ax3.plot(iterations, relative_errors, 's-', color=colors[name], label=name, linewidth=2)
    
    ax3.set_xlabel('Iteration')
    ax3.set_ylabel('Relative Error (%)')
    ax3.set_title('Relative Error Convergence')
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    
    # Plot 4: Step sizes
    for name, ppm, history in methods_data:
        if history:
            if len(history) > 1:
                iterations = range(1, len(history))
                step_sizes = [abs(history[i+1][0] - history[i][0]) for i in range(len(history)-1)]
                ax4.plot(iterations, step_sizes, '^-', color=colors[name], label=name, linewidth=2)
    
    ax4.set_xlabel('Iteration')
    ax4.set_ylabel('Step Size (ppm)')
    ax4.set_title('Step Size Evolution')
    ax4.legend()
    ax4.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    print(f"Detailed convergence plots saved as '{output_file}'")
    plt.show()

# ===============================================================
# Comparison and Analysis
# ===============================================================
def compare_optimization_methods(ppm_start, k_target):
    """Compare all three optimization methods"""
    print("=" * 80)
    print("COMPARING OPTIMIZATION METHODS FOR BORON CONCENTRATION SEARCH")
    print(f"Target k_eff: {k_target}, Initial guess: {ppm_start} ppm")
    print("=" * 80)
    
    # Method 1: Gradient-based (analytical derivatives)
    grad_ppm, grad_history = gradient_based_search(ppm_start, k_target, max_iter=6)
    
    # Method 2: Finite difference
    fd_ppm, fd_history = finite_difference_search(ppm_start, k_target, max_iter=6)
    
    # Method 3: Gradient-free
    free_ppm, free_history = gradient_free_search(ppm_start, k_target, max_iter=8)
    
    # Results comparison
    print("\n" + "=" * 80)
    print("FINAL RESULTS COMPARISON")
    print("=" * 80)
    
    methods = [
        ("Analytical Gradient", grad_ppm, grad_history),
        ("Finite Difference", fd_ppm, fd_history), 
        ("Gradient-Free", free_ppm, free_history)
    ]
    
    best_method = None
    best_error = float('inf')
    
    for name, ppm, history in methods:
        if history:
            final_k = history[-1][1]
            final_err = abs(history[-1][2])
            iterations = len(history)
            
            print(f"\n{name}:")
            print(f"  Final ppm: {ppm:.1f}")
            print(f"  Final k_eff: {final_k:.6f}")
            print(f"  Final error: {final_err:.6f}")
            print(f"  Iterations: {iterations}")
            
            if final_err < best_error:
                best_error = final_err
                best_method = name
    
    if best_method:
        print(f"\n★ BEST METHOD: {best_method} (error = {best_error:.6f})")
    
    # Convergence speed analysis
    print(f"\nCONVERGENCE SPEED ANALYSIS:")
    tolerance_levels = [0.05, 0.02, 0.01]  # 5%, 2%, 1% tolerance
    for name, ppm, history in methods:
        if history:
            print(f"\n{name}:")
            for tol_level in tolerance_levels:
                iterations_to_tolerance = None
                for i, (_, k, err) in enumerate(history):
                    if abs(err) < tol_level:
                        iterations_to_tolerance = i + 1
                        break
                if iterations_to_tolerance:
                    print(f"  Reached {tol_level*100:.0f}% tolerance in {iterations_to_tolerance} iterations")
                else:
                    print(f"  Did not reach {tol_level*100:.0f}% tolerance")
    
    # Generate plots
    try:
        plot_convergence_comparison(methods, k_target)
        plot_detailed_convergence(methods, k_target)
    except Exception as e:
        print(f"\nPlotting failed: {e}")
        print("Please install matplotlib: pip install matplotlib")
    
    return methods

# ===============================================================
# Main execution
# ===============================================================
if __name__ == '__main__':
    # Parameters
    ppm_start = 1000.0
    k_target = 0.95  # Subcritical target
    
    print("GRADIENT-BASED OPTIMIZATION DEMONSTRATION")
    print("=========================================")
    print("This demo compares three optimization methods:")
    print("1. Analytical Gradient: Uses OpenMC's built-in derivative tallies")
    print("2. Finite Difference: Estimates gradient via perturbation")  
    print("3. Gradient-Free: Uses heuristic step sizes without gradient info")
    print(f"\nTarget: k_eff = {k_target} (subcritical configuration)")
    print(f"Starting from: {ppm_start} ppm boron")
    print("\nThe gradient methods should converge faster by using local sensitivity information!")
    
    try:
        results = compare_optimization_methods(ppm_start, k_target)
        
        print("\n" + "=" * 80)
        print("KEY INSIGHTS:")
        print("=" * 80)
        print("• Analytical gradients provide exact local sensitivity information")
        print("• Finite differences approximate gradients but require extra simulations")  
        print("• Gradient-free methods are more robust but may converge slower")
        print("• For reactor physics, gradient methods can significantly reduce")
        print("  the number of expensive Monte Carlo simulations needed")
        
    except Exception as e:
        print(f"\nOptimization failed: {e}")
        print("This might be due to OpenMC simulation issues or file conflicts.")